# RFDC V4 -- Row Program, No-Ethernet Timing Bench

Purpose: isolate and *measure* the one thing that's actually new and unproven compared to
`RFDC_V4_Pump_Pi2_Probe_TTL_Loop_Qualification_Ethernet.ipynb` -- mid-run, Python-timed TX DMA
refills -- without any of the other new machinery (Ethernet, background thread, double
buffering) in the way.

**What changed vs `RFDC_V4_RowProgram_Design_FIXED_2_.ipynb`:**
- No Ethernet, no socket, no background sender thread. Nothing competes with the run loop
  for the GIL during a shot.
- No RX double buffering -- one buffer per RX lane, reused every shot, exactly like the
  qualification notebook. There's nothing to overlap without Ethernet, so it bought nothing
  here and only added PS DDR4 usage.
- **One `allocate()` per TX lane, not one per chunk.** The whole lane's waveform (all reps
  concatenated) is quantized once into a single PL buffer; each refill is a *slice* of that
  one buffer (`buf[c0:c1]`), not a separate allocation. This is the fix for "why 20 buffers" --
  chunking the DMA *transfer* doesn't require chunking the *allocation*.
- **Every refill is timestamped** against a model of when the row that consumes it actually
  starts (derived from the committed row table's own `gap_cycles`), so instead of guessing
  whether a 2 ms window is enough, this notebook prints the real margin -- or a negative
  number, in red, if a refill would have missed its deadline.
- A **standalone `transfer()` call-overhead benchmark runs first**, before anything is armed,
  so you have a real number for "how long does issuing a refill actually take on this board"
  instead of the placeholder guess (`TRANSFER_CALL_OVERHEAD_S_ESTIMATE`) that the earlier
  notebook only warned about.

**Status: not yet run against hardware.** Everything below follows from the same VHDL/register
map already validated in the qualification notebook; only the refill polling loop and the
single-buffer-with-slices pattern are new here.

## Cell 1 -- Overlay + imports (only this)

In [1]:
# ================= OVERLAY + IMPORTS (run this once) =================
import time
import numpy as np
import xrfclk
import xrfdc
from pynq import Overlay, allocate

BITFILE = "./final.bit"   # confirm this is the 256-row Pulse_Sequencer build
base = Overlay(BITFILE)
print("Overlay loaded:", BITFILE)


Overlay loaded: ./final.bit


## Cell 2 -- User configuration

In [2]:
# ================= USER CONFIG =================
AXIS_BEAT_HZ = 15.36e6
SAMPLES_PER_BEAT = 8
SEQ_CLK_HZ = 99_999_985.0
FS_HZ = 122.88e6
AMPLITUDE = 32760

DAC_A_NCO_MHZ = 10.0
DAC_B_NCO_MHZ = 0.2

DMA_MAX_BYTES = (1 << 26) - 1
CAP_MAX_S = 0.260
TX_MAX_S = 0.136

LOOP_COUNT = 5           # keep small for a bench run; bump once margins look safe
DEBUG_VERBOSE = True

EXPECTED_IP_PATHS = {
    "sequencer": "radio/AXI_Pulse_Sequencer_0",
    "tx_gate_b": "radio/AXI_TX_Multi_Gate_0",
    "tx_gate_a": "radio/AXI_TX_Multi_Gate_1",
    "cap_gate_b": "radio/receiver/channel_20/AXI_Capture_Gate_0",
    "cap_gate_a": "radio/receiver/channel_21/AXI_Capture_Gate_0",
    "rx_dma_b": "radio/receiver/channel_20/axi_dma_real",
    "rx_dma_a": "radio/receiver/channel_21/axi_dma_real",
    "tx_dma_b": "radio/axi_dma_dac_0",
    "tx_dma_a": "radio/axi_dma_dac_1",
}
print("Config loaded.")


Config loaded.


## Cell 3 -- IP resolution

In [3]:
# ================= IP RESOLUTION =================
def get_by_path(root, path):
    obj = root
    for part in path.split('/'):
        obj = getattr(obj, part)
    return obj

missing = [p for p in EXPECTED_IP_PATHS.values() if p not in base.ip_dict]
if missing:
    print("Missing expected HWH paths:")
    for p in missing:
        print("  ", p)
    raise KeyError("The loaded .hwh does not match the validated A/B topology.")

seq   = get_by_path(base, EXPECTED_IP_PATHS['sequencer'])
tx_b  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_b'])
tx_a  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_a'])
cap_b = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_b'])
cap_a = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_a'])
dma_rb = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_b'])
dma_ra = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_a'])
dma_tb = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_b'])
dma_ta = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_a'])

rfdc = get_by_path(base, 'radio/rfdc')
dac_b = rfdc.dac_tiles[0].blocks[0]
dac_a = rfdc.dac_tiles[2].blocks[0]
adc_b = rfdc.adc_tiles[2].blocks[0]
adc_a = rfdc.adc_tiles[2].blocks[1]
print('IP resolved.')


IP resolved.


## Cell 4 -- Register map + unit-conversion helpers

In [4]:
# ================= REGISTER MAP =================
SEQ_ROW_SEL=0x00; SEQ_MASK=0x04; SEQ_GAP=0x08
SEQ_DUR0=0x0C; SEQ_DUR1=0x10; SEQ_DUR2=0x14; SEQ_DUR3=0x18
SEQ_COMMIT=0x1C; SEQ_TABLE_LEN=0x20; SEQ_ENABLE=0x24
SEQ_RESET=0x28; SEQ_ROW_PTR=0x2C; SEQ_CDC_OVERRUN=0x30
SEQ_TTL_ARM=0x34; SEQ_TTL_MODE=0x38; SEQ_TTL_STATUS=0x3C
SEQ_TTL_STATUS_CLEAR=0x40; SEQ_TTL_EDGE_COUNT=0x44
TTL_ARMED=1<<0; TTL_RUNNING=1<<1; TTL_RUN_DONE=1<<2

TXM_SEG_SEL=0x00; TXM_ACTIVE_LEN=0x04; TXM_GAP_LEN=0x08; TXM_COMMIT=0x0C
TXM_NUM_SEGMENTS=0x10; TXM_STATUS=0x1C
TX_BUSY=1<<0; TX_OVERRUN=1<<1

CAP_LENGTH=0x00; CAP_STATUS=0x08; CAP_CLEAR=0x0C
CAP_BUSY=1<<0; CAP_OVERFLOW=1<<1

DMASR_HALTED=1<<0; DMASR_IDLE=1<<1; DMASR_ERR_MASK=(1<<4)|(1<<5)|(1<<6)

LANE_TX_B, LANE_TX_A, LANE_RX_B, LANE_RX_A = 0, 1, 2, 3
LANE_BIT = {"TX_B": LANE_TX_B, "TX_A": LANE_TX_A, "RX_B": LANE_RX_B, "RX_A": LANE_RX_A}
TX_LANES = ("TX_A", "TX_B")
RX_LANES = ("RX_A", "RX_B")

def beats_for(seconds):
    return max(1, int(round(seconds * AXIS_BEAT_HZ)))

def samples_for(beats):
    return int(beats) * SAMPLES_PER_BEAT

def seq_cycles_for(seconds):
    return max(1, int(round(seconds * SEQ_CLK_HZ)))

def pack_iq(i, q):
    if len(i) != len(q):
        raise ValueError('I/Q length mismatch')
    out = np.empty(2*len(i), dtype=np.int16)
    out[0::2] = i
    out[1::2] = q
    return out
print('Register map ready.')


Register map ready.


## Cell 5 -- Waveform primitives

Unchanged from the row-program design: `chirp()` (sine is `f0==f1`), `zeros()`,
`envelope()`, `waveform(*parts)`, `quantize_iq()` (final step only), `chirp_train()`.

In [5]:
# ================= WAVEFORM PRIMITIVES =================
def chirp(f0_mhz, f1_mhz, duration_s, nco_mhz):
    n = samples_for(beats_for(duration_s))
    t = np.arange(n, dtype=np.float64) / FS_HZ
    f0 = (f0_mhz - nco_mhz) * 1e6
    f1 = (f1_mhz - nco_mhz) * 1e6
    k = (f1 - f0) / duration_s if duration_s > 0 else 0.0
    ph = 2*np.pi*(f0*t + 0.5*k*t*t)
    return np.exp(1j*ph).astype(np.complex128)

def zeros(duration_s):
    n = samples_for(beats_for(duration_s))
    return np.zeros(n, dtype=np.complex128)

def envelope(arr, kind="gaussian", **params):
    n = len(arr)
    if kind == "gaussian":
        sigma = params.get("sigma", 0.25) * n
        x = np.arange(n) - (n-1)/2.0
        win = np.exp(-0.5*(x/sigma)**2)
    else:
        raise ValueError(f"unknown envelope kind: {kind}")
    return arr * win

def waveform(*parts):
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.complex128)

def quantize_iq(arr):
    i = np.round(AMPLITUDE * arr.real).astype(np.int16)
    q = np.round(AMPLITUDE * arr.imag).astype(np.int16)
    return pack_iq(i, q)

def chirp_train(freqs_mhz, pulse_s, gap_s, nco_mhz):
    parts = []
    for idx, f in enumerate(freqs_mhz):
        parts.append(chirp(f, f, pulse_s, nco_mhz))
        if idx < len(freqs_mhz) - 1:
            parts.append(zeros(gap_s))
    return waveform(*parts)

print('Waveform primitives ready.')


Waveform primitives ready.


## Cell 6 -- Row helpers: `repeat()` and `print_program()`

In [6]:
# ================= ROW HELPERS =================
def repeat(rows, n):
    return list(rows) * n

def _row_lanes(row):
    lanes = []
    for lane in TX_LANES:
        if lane in row.get("tx", {}):
            lanes.append(lane)
    for lane in RX_LANES:
        if lane in row.get("rx", {}):
            lanes.append(lane)
    return lanes

def print_program(rows):
    print(f"{'row':>4}  {'gap_s':>10}  {'lanes (dur_s)':<50} {'refill'}")
    for idx, row in enumerate(rows):
        parts = []
        for lane in _row_lanes(row):
            if lane in TX_LANES:
                arr = row['tx'][lane]
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms({len(arr)}smp)")
            else:
                dur_s = row['rx'][lane]
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms")
        refill = ','.join(row.get('refill', [])) or '-'
        print(f"{idx:>4}  {row['gap_s']*1e3:>9.4f}ms  {', '.join(parts):<50} {refill}")

print('repeat() and print_program() ready.')


repeat() and print_program() ready.


## Cell 7 -- The program (same sequence as the design that hung, unchanged on purpose)

Kept identical to `RFDC_V4_RowProgram_Design_FIXED_2_.ipynb`'s state-prep / spectroscopy /
detection example so this bench is testing the same timing that failed, not an easier case.

In [7]:
# ================= PROGRAM DEFINITION =================
ROW_MARGIN_S = 5e-6

def gap_after(*durations_s):
    return max(durations_s) + ROW_MARGIN_S

state_prep_dacA = waveform(chirp(15, 15, 1e-3, DAC_A_NCO_MHZ), chirp(5, 10, 1e-3, DAC_A_NCO_MHZ))
state_prep_dacB = chirp(5, 25, 20e-3, DAC_B_NCO_MHZ)

state_prep_block = [
    {"gap_s": gap_after(2e-3), "tx": {"TX_A": state_prep_dacA}},
    {"gap_s": gap_after(20e-3), "tx": {"TX_B": state_prep_dacB}, "refill": ["TX_B"]},
]
state_prep_block_final = [
    state_prep_block[0],
    {"gap_s": gap_after(20e-3), "tx": {"TX_B": state_prep_dacB}},
]
STATE_PREP_REPS = 20
state_prep_rows = repeat(state_prep_block, STATE_PREP_REPS - 1) + state_prep_block_final

SPECTROSCOPY_S = 5e-3
spectroscopy_rows = [
    {"gap_s": gap_after(SPECTROSCOPY_S), "tx": {}, "rx": {}},
]

probe = chirp_train([190, 191, 192, 193, 194], pulse_s=10e-6, gap_s=1e-6, nco_mhz=DAC_A_NCO_MHZ)
CAPTURE_S = len(probe) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ

detection_rows = [
    {"gap_s": gap_after(CAPTURE_S), "tx": {"TX_A": probe}, "rx": {"RX_A": CAPTURE_S, "RX_B": CAPTURE_S}},
]

rows = state_prep_rows + spectroscopy_rows + detection_rows
print_program(rows)


 row       gap_s  lanes (dur_s)                                      refill
   0     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   1    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   2     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   3    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   4     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   5    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   6     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   7    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
   8     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   9    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
  10     2.0050ms  TX_A:2.0000ms(245760smp)                           -
  11    20.0050ms  TX_B:20.0000ms(2457600smp)                         TX_B
  12     2.0050ms  TX_A:2.0000ms(245760smp

## Cell 8 -- Compile: derive mask/dur, ONE buffer per lane, refill deadlines

Same mask/DUR derivation as before. New: `row_start_s[idx]` models the wall-clock time (from
`TTL_ARM`) each row *starts*, purely from the committed `gap_cycles` -- this is what lets the
run loop print a real number for "how much margin did this refill actually have" instead of
just hoping a 2 ms window was enough.

In [8]:
# ================= COMPILE PROGRAM =================
def compile_program(rows):
    table = []
    tx_chunks = {lane: [] for lane in TX_LANES}
    tx_chunk_bounds = {lane: [0] for lane in TX_LANES}
    rx_total_s = {lane: 0.0 for lane in RX_LANES}
    refill_points = []   # (row_idx, lane, chunk_idx)
    lane_chunk_counter = {lane: 1 for lane in TX_LANES}

    for idx, row in enumerate(rows):
        mask = 0
        dur = {}
        for lane in TX_LANES:
            arr = row.get("tx", {}).get(lane)
            if arr is not None:
                mask |= (1 << LANE_BIT[lane])
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                dur[lane] = beats_for(dur_s)
                tx_chunks[lane].append(arr)
        for lane in RX_LANES:
            dur_s = row.get("rx", {}).get(lane)
            if dur_s:
                mask |= (1 << LANE_BIT[lane])
                dur[lane] = beats_for(dur_s)
                rx_total_s[lane] += dur_s
        for lane in row.get("refill", []):
            refill_points.append((idx, lane, lane_chunk_counter[lane]))
            lane_chunk_counter[lane] += 1
            tx_chunk_bounds[lane].append(sum(len(a) for a in tx_chunks[lane]))
        table.append({"mask": mask, "gap_cycles": seq_cycles_for(row["gap_s"]), "dur": dur})

    tx_full = {lane: (np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.complex128))
               for lane, chunks in tx_chunks.items()}
    for lane in TX_LANES:
        tx_chunk_bounds[lane].append(len(tx_full[lane]))

    # Model row start times purely from the committed gap_cycles (time at which SEQ_ROW_PTR
    # becomes idx, measured from TTL_ARM). This is a model of the hardware schedule, used only
    # to print real margins during the run -- it is not read back from hardware.
    row_start_s = [0.0]
    for row in table:
        row_start_s.append(row_start_s[-1] + row["gap_cycles"] / SEQ_CLK_HZ)

    # For each refill point, find the deadline: the start time of the NEXT row (after the
    # refill's own row) where that lane fires again -- that's the row the new chunk must be
    # loaded before.
    refill_deadlines = []   # (row_idx, lane, chunk_idx, deadline_s)
    for row_idx, lane, chunk_idx in refill_points:
        deadline = None
        for j in range(row_idx + 1, len(rows)):
            if lane in rows[j].get("tx", {}):
                deadline = row_start_s[j]
                break
        refill_deadlines.append((row_idx, lane, chunk_idx, deadline))

    return table, tx_full, tx_chunk_bounds, refill_deadlines, rx_total_s, row_start_s

table, tx_full, tx_chunk_bounds, refill_deadlines, rx_total_s, row_start_s = compile_program(rows)

if len(table) > 256:
    raise ValueError(f"Program has {len(table)} rows, exceeds the 256-row Pulse_Sequencer build.")

for lane in TX_LANES:
    print(f"{lane}: {len(tx_full[lane]):,} samples total across the whole program")
for lane, secs in rx_total_s.items():
    if secs > CAP_MAX_S:
        raise RuntimeError(f"{lane}: total capture {secs*1e3:.1f} ms exceeds the {CAP_MAX_S*1e3:.0f} ms RX DMA limit.")
    print(f"{lane}: {secs*1e3:.3f} ms total capture this program")

print()
print("Refill schedule (modeled from committed gap_cycles, not yet measured against real hardware):")
for row_idx, lane, chunk_idx, deadline in refill_deadlines:
    window_s = deadline - row_start_s[row_idx + 1] if deadline is not None else None
    print(f"  row {row_idx:>3}: load {lane} chunk {chunk_idx} -- consumed at row starting "
          f"{deadline*1e3:.4f} ms (window after this row ends: "
          f"{'n/a' if window_s is None else f'{window_s*1e3:.4f} ms'})")

for lane in TX_LANES:
    bounds = tx_chunk_bounds[lane]
    for c0, c1 in zip(bounds[:-1], bounds[1:]):
        chunk_bytes = (c1 - c0) * 4
        if chunk_bytes > DMA_MAX_BYTES:
            raise RuntimeError(f"{lane}: a chunk between samples {c0}-{c1} is {chunk_bytes:,} bytes, "
                                f"exceeds the {DMA_MAX_BYTES:,}-byte single-DMA-transfer limit.")
    chunk_sizes = [(b1-b0)*4 for b0, b1 in zip(bounds[:-1], bounds[1:])]
    print(f"{lane}: {len(bounds)-1} chunk(s), sizes(bytes)={chunk_sizes}")

print("PASS: program compiled and passes capacity/consistency checks.")


TX_A: 4,921,840 samples total across the whole program
TX_B: 49,152,000 samples total across the whole program
RX_A: 0.054 ms total capture this program
RX_B: 0.054 ms total capture this program

Refill schedule (modeled from committed gap_cycles, not yet measured against real hardware):
  row   1: load TX_B chunk 1 -- consumed at row starting 24.0150 ms (window after this row ends: 2.0050 ms)
  row   3: load TX_B chunk 2 -- consumed at row starting 46.0250 ms (window after this row ends: 2.0050 ms)
  row   5: load TX_B chunk 3 -- consumed at row starting 68.0350 ms (window after this row ends: 2.0050 ms)
  row   7: load TX_B chunk 4 -- consumed at row starting 90.0450 ms (window after this row ends: 2.0050 ms)
  row   9: load TX_B chunk 5 -- consumed at row starting 112.0550 ms (window after this row ends: 2.0050 ms)
  row  11: load TX_B chunk 6 -- consumed at row starting 134.0650 ms (window after this row ends: 2.0050 ms)
  row  13: load TX_B chunk 7 -- consumed at row starting 156.

## Cell 9 -- Standalone `transfer()` call-overhead benchmark (run before anything is armed)

Measures pure Python/PYNQ/AXI-Lite overhead of issuing a refill-sized `transfer()` call, with
the sequencer not even running (so nothing drains the buffer and this is purely the call
itself). This replaces the placeholder `TRANSFER_CALL_OVERHEAD_S_ESTIMATE` guess with a real
number for this board. `ch.stop()` resets the channel to halted between iterations so each
call starts from the same state `_try_refill()` would see mid-run.

In [9]:
# ================= transfer() CALL OVERHEAD BENCHMARK =================
N_BENCH = 50
bench_chunk = quantize_iq(state_prep_dacB)   # same size/shape as a real TX_B refill chunk
bench_buf = allocate(shape=bench_chunk.shape, dtype=np.int16)
bench_buf[:] = bench_chunk
bench_buf.flush()

ch = dma_tb.sendchannel
times_s = []
for i in range(N_BENCH):
    ch.stop()
    st = ch._mmio.read(int(ch._offset) + 0x04)
    if st & DMASR_ERR_MASK:
        raise RuntimeError(f"bench: TX-B DMA error before iteration {i}: 0x{st:08x}")
    t0 = time.perf_counter()
    ch.transfer(bench_buf)
    t1 = time.perf_counter()
    times_s.append(t1 - t0)

ch.stop()
times_ms = sorted(t*1e3 for t in times_s)
print(f"transfer() call overhead over {N_BENCH} calls (buffer={bench_buf.nbytes:,} bytes):")
print(f"  min:    {times_ms[0]:.4f} ms")
print(f"  median: {times_ms[len(times_ms)//2]:.4f} ms")
print(f"  max:    {times_ms[-1]:.4f} ms")
print()
tightest_window_ms = min((deadline - row_start_s[row_idx+1]) for row_idx,_,_,deadline in refill_deadlines
                          if deadline is not None) * 1e3
print(f"Tightest modeled refill window in this program: {tightest_window_ms:.4f} ms")
if times_ms[-1] > tightest_window_ms:
    print("WARNING: worst-case transfer() overhead alone exceeds the tightest refill window. "
          "This program's refill timing is not safe as written -- widen the gap or reduce reps "
          "before trusting the run loop below.")
else:
    print("PASS: worst-case transfer() overhead fits inside the tightest refill window with "
          f"{tightest_window_ms - times_ms[-1]:.4f} ms to spare. This does not yet include "
          "row_ptr polling latency -- see the run loop's measured margins below for the real number.")


RuntimeError: DMA channel not started

## Cell 10 -- RFDC NCO setup + DMA buffers (one buffer per TX lane, sliced for chunks)

The fix for "why 20 buffers": each lane's whole waveform is quantized once into a single PL
buffer; refill chunks are slices of that one buffer, not separate allocations. RX is a single
buffer per lane (no double buffering -- nothing to overlap without Ethernet).

In [ ]:
# ================= RFDC NCO SETUP =================
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    ms['Freq'] = float(freq)
    ms['PhaseOffset'] = 0.0
    ms['EventSource'] = 2
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    assert abs(float(dac.MixerSettings['Freq']) - freq) < 1e-6
print('PASS: DAC mixer state matches config.')

# ================= DMA BUFFERS (single allocation per lane) =================
tx_full_buf = {}     # lane -> one PynqBuffer holding the WHOLE quantized waveform
tx_chunk_view = {}    # lane -> list of slices (views) into tx_full_buf[lane], one per chunk
for lane in TX_LANES:
    quantized = quantize_iq(tx_full[lane])           # one quantize_iq() call for the whole lane
    buf = base.device.get_memory_by_idx(1).allocate(shape=quantized.shape, dtype=np.int16)
    buf[:] = quantized
    buf.flush()                                       # one flush for the whole buffer
    tx_full_buf[lane] = buf

    bounds = tx_chunk_bounds[lane]
    views = [buf[2*c0:2*c1] for c0, c1 in zip(bounds[:-1], bounds[1:])]
    tx_chunk_view[lane] = views

    # Sanity-check the key assumption this whole design rests on: that a slice of a PynqBuffer
    # reports the correct physical address (base + byte offset), so transfer()ing a slice
    # actually points the DMA at the right bytes instead of silently re-sending chunk 0.
    base_pa = int(buf.physical_address)
    for k, (c0, c1) in enumerate(zip(bounds[:-1], bounds[1:])):
        expected_pa = base_pa + 2*c0*2   # 2 int16/sample * 2 bytes/int16
        actual_pa = int(views[k].physical_address)
        if actual_pa != expected_pa:
            raise RuntimeError(f"{lane} chunk {k}: slice physical_address 0x{actual_pa:x} != "
                                f"expected 0x{expected_pa:x}. Buffer slicing does not behave as "
                                f"assumed on this PYNQ version -- do not trust sliced transfers; "
                                f"fall back to one allocate() per chunk instead.")
    print(f"{lane}: 1 buffer allocated ({buf.nbytes:,} bytes), {len(views)} chunk view(s), "
          f"physical addresses verified.")

rx_buffers = {}
for lane in RX_LANES:
    capture_beats = beats_for(rx_total_s[lane])
    capture_samples = samples_for(capture_beats)
    rx_buffers[lane] = allocate(shape=(capture_samples,), dtype=np.int16)
    print(f"{lane}: RX buffer {rx_buffers[lane].nbytes:,} bytes (single, reused every shot).")

print('PASS: DMA buffers allocated.')


## Cell 11 -- Commit the compiled table to the sequencer

In [ ]:
# ================= COMMIT TABLE TO HARDWARE =================
def commit_table(table):
    seq.mmio.write(SEQ_TTL_MODE, 0)
    seq.mmio.write(SEQ_ENABLE, 0)
    seq.mmio.write(SEQ_RESET, 1)
    time.sleep(100e-6)
    for idx, row in enumerate(table):
        seq.mmio.write(SEQ_ROW_SEL, idx)
        seq.mmio.write(SEQ_MASK, row["mask"])
        seq.mmio.write(SEQ_GAP, row["gap_cycles"])
        seq.mmio.write(SEQ_DUR0, row["dur"].get("TX_B", 0))
        seq.mmio.write(SEQ_DUR1, row["dur"].get("TX_A", 0))
        seq.mmio.write(SEQ_DUR2, row["dur"].get("RX_B", 0))
        seq.mmio.write(SEQ_DUR3, row["dur"].get("RX_A", 0))
        seq.mmio.write(SEQ_COMMIT, 1)
    seq.mmio.write(SEQ_TABLE_LEN, len(table))
    print(f"PASS: committed {len(table)} rows to the sequencer.")

commit_table(table)


## Cell 12 -- DMA/loop helpers + `dump_state()` (no Ethernet, no threading)

In [ ]:
# ================= DMA / LOOP HELPERS =================
def dma_status(ch): return int(ch._mmio.read(int(ch._offset)+0x04))
def dma_flags(v):
    out = ['halted' if v&1 else 'running']
    if v&2: out.append('idle')
    if v&0x10: out.append('INTERNAL_ERR')
    if v&0x20: out.append('SLAVE_ERR')
    if v&0x40: out.append('DECODE_ERR')
    return '|'.join(out)

def ensure_running(ch, label):
    st = dma_status(ch)
    if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error before shot: 0x{st:08x} ({dma_flags(st)})')
    if st & DMASR_HALTED:
        ch.start(); t0 = time.perf_counter()
        while time.perf_counter()-t0 < 0.2:
            st = dma_status(ch)
            if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error restarting: 0x{st:08x} ({dma_flags(st)})')
            if not (st & DMASR_HALTED): return
            time.sleep(1e-5)
        raise TimeoutError(f'{label}: DMA did not become running: 0x{st:08x}')

def wait_dma_idle(ch, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = dma_status(ch)
        if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error: 0x{st:08x} ({dma_flags(st)})')
        if st & DMASR_IDLE: return
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: DMA not idle: 0x{st:08x} ({dma_flags(st)})')
        time.sleep(50e-6)

def wait_cap_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(CAP_STATUS))
        if not (st & CAP_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: Capture_Gate busy: 0x{st:08x}')
        time.sleep(100e-6)

def wait_tx_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(TXM_STATUS))
        if not (st & TX_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: TX gate busy: 0x{st:08x}')
        time.sleep(20e-6)

def discard_rx(buf):
    inv = getattr(buf, 'invalidate', None)
    if inv:
        try: inv()
        except Exception: pass

def ttl_arm(s): s.mmio.write(SEQ_TTL_ARM, 1)

def dump_state(label=""):
    print(f"---- dump_state: {label} ----")
    try:
        print(f"  seq:   ROW_PTR={int(seq.mmio.read(SEQ_ROW_PTR))} "
              f"TTL_STATUS=0x{int(seq.mmio.read(SEQ_TTL_STATUS)):x} "
              f"CDC_OVERRUN=0x{int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF:x}")
    except Exception as e:
        print(f"  seq: <read failed: {e}>")
    for name, g in [("tx_a", tx_a), ("tx_b", tx_b)]:
        try:
            st = int(g.mmio.read(TXM_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&TX_BUSY)}, overrun={bool(st&TX_OVERRUN)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, g in [("cap_a", cap_a), ("cap_b", cap_b)]:
        try:
            st = int(g.mmio.read(CAP_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&CAP_BUSY)}, overflow={bool(st&CAP_OVERFLOW)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, ch in [("RX-A", dma_ra.recvchannel), ("RX-B", dma_rb.recvchannel),
                      ("TX-A", dma_ta.sendchannel), ("TX-B", dma_tb.sendchannel)]:
        try:
            st = dma_status(ch)
            print(f"  {name}: 0x{st:08x} ({dma_flags(st)}) transferred={int(ch.transferred)}")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    print("-" * (18 + len(label)))

print('DMA/loop helpers ready.')


## Cell 13 -- Run loop

Synchronous, single-buffered, no Ethernet. Every refill logs the *measured* margin: modeled
deadline (from Cell 8) minus actual wall-clock time the `transfer()` call returned, both
relative to this shot's own `TTL_ARM`. A negative margin means that refill would have missed
its window -- exactly the failure mode from the earlier run, now visible instead of inferred.

In [ ]:
# ================= RUN LOOP =================
def prepare_shot():
    wait_cap_idle(cap_a, "RX_A", 2.0); wait_cap_idle(cap_b, "RX_B", 2.0)
    wait_tx_idle(tx_a, "TX_A", 1.0); wait_tx_idle(tx_b, "TX_B", 1.0)
    cap_a.mmio.write(CAP_CLEAR, CAP_OVERFLOW); cap_b.mmio.write(CAP_CLEAR, CAP_OVERFLOW)
    cap_a.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_A"]))
    cap_b.mmio.write(CAP_LENGTH, beats_for(rx_total_s["RX_B"]))
    tx_a.mmio.write(TXM_STATUS, TX_OVERRUN); tx_b.mmio.write(TXM_STATUS, TX_OVERRUN)
    seq.mmio.write(SEQ_ENABLE, 0); seq.mmio.write(SEQ_RESET, 1); time.sleep(100e-6)
    seq.mmio.write(SEQ_CDC_OVERRUN, 0xF); seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1)
    seq.mmio.write(SEQ_TTL_MODE, 1)
    for ch, label in [(dma_ra.recvchannel,'RX-A'), (dma_rb.recvchannel,'RX-B'),
                       (dma_ta.sendchannel,'TX-A'), (dma_tb.sendchannel,'TX-B')]:
        ensure_running(ch, label)

def arm_shot():
    dma_ra.recvchannel.transfer(rx_buffers["RX_A"])
    dma_rb.recvchannel.transfer(rx_buffers["RX_B"])
    dma_ta.sendchannel.transfer(tx_chunk_view["TX_A"][0])
    dma_tb.sendchannel.transfer(tx_chunk_view["TX_B"][0])
    time.sleep(5e-3)
    for ch, label in [(dma_ra.recvchannel,'RX-A'), (dma_rb.recvchannel,'RX-B'),
                       (dma_ta.sendchannel,'TX-A'), (dma_tb.sendchannel,'TX-B')]:
        st = dma_status(ch)
        if st & (DMASR_ERR_MASK | DMASR_HALTED):
            raise RuntimeError(f'{label}: bad after arm 0x{st:08x} ({dma_flags(st)})')

DMA_LANE_TO_CHANNEL = {"TX_A": dma_ta.sendchannel, "TX_B": dma_tb.sendchannel}

def _try_refill(row_idx, lane, chunk_idx, deadline, t_arm, margins):
    if chunk_idx >= len(tx_chunk_view[lane]):
        return True
    ch = DMA_LANE_TO_CHANNEL[lane]
    st = dma_status(ch)
    if st & DMASR_ERR_MASK:
        raise RuntimeError(f'{lane}: DMA error before refill (row {row_idx}): 0x{st:08x} ({dma_flags(st)})')
    if not (st & DMASR_IDLE):
        return False
    ch.transfer(tx_chunk_view[lane][chunk_idx])
    t_issued = time.perf_counter()
    if deadline is not None:
        margin_s = deadline - (t_issued - t_arm)
        margins.append(margin_s)
        flag = "" if margin_s > 0 else "  <<< MISSED DEADLINE"
        if DEBUG_VERBOSE:
            print(f"  refill {lane} chunk {chunk_idx} (row {row_idx}): margin {margin_s*1e3:+.4f} ms{flag}")
    return True

def run_shot(k):
    t0 = time.perf_counter()
    try:
        prepare_shot(); arm_shot()
    except Exception:
        dump_state(f"shot {k} prepare/arm failure")
        raise

    pending = [(ri, ln, ci, dl) for ri, ln, ci, dl in refill_deadlines]
    t_arm = time.perf_counter()
    ttl_arm(seq)
    margins = []

    deadline_wall = time.perf_counter() + 5.0
    while True:
        row_ptr = int(seq.mmio.read(SEQ_ROW_PTR))
        while pending and pending[0][0] <= row_ptr:
            row_idx, lane, chunk_idx, deadline = pending[0]
            if _try_refill(row_idx, lane, chunk_idx, deadline, t_arm, margins):
                pending.pop(0)
            else:
                break
        cdc = int(seq.mmio.read(SEQ_CDC_OVERRUN) & 0xF)
        if cdc:
            dump_state(f"shot {k} CDC overrun")
            raise RuntimeError(f'shot {k}: CDC overrun 0x{cdc:x}')
        if int(seq.mmio.read(SEQ_TTL_STATUS)) & TTL_RUN_DONE: break
        if time.perf_counter() > deadline_wall:
            dump_state(f"shot {k} RUN_DONE timeout")
            raise TimeoutError(f'shot {k}: RUN_DONE timeout, row_ptr={row_ptr}')
        time.sleep(0.5e-3)

    seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1)
    row = int(seq.mmio.read(SEQ_ROW_PTR))

    tout = max(2.0, max(rx_total_s.values()) + 1.0)
    try:
        wait_cap_idle(cap_a, "RX_A", tout); wait_cap_idle(cap_b, "RX_B", tout)
        wait_dma_idle(dma_ra.recvchannel, 'RX-A', tout); wait_dma_idle(dma_rb.recvchannel, 'RX-B', tout)
        wait_dma_idle(dma_ta.sendchannel, 'TX-A', 2.0); wait_dma_idle(dma_tb.sendchannel, 'TX-B', 2.0)
    except Exception:
        dump_state(f"shot {k} post-run wait failure")
        raise
    dma_ra.recvchannel.wait(); dma_rb.recvchannel.wait(); dma_ta.sendchannel.wait(); dma_tb.sendchannel.wait()

    ca = int(cap_a.mmio.read(CAP_STATUS)); cb = int(cap_b.mmio.read(CAP_STATUS))
    overflow = bool(ca & CAP_OVERFLOW) or bool(cb & CAP_OVERFLOW)
    discard_rx(rx_buffers["RX_A"]); discard_rx(rx_buffers["RX_B"])

    return {'shot': k, 'row': row, 'overflow': overflow,
            'elapsed_s': time.perf_counter()-t0,
            'min_margin_s': (min(margins) if margins else None)}

print('Run-loop functions ready.')


## Cell 14 -- Run + summary

In [ ]:
# ================= RUN =================
results = []
for k in range(1, LOOP_COUNT+1):
    try:
        r = run_shot(k); results.append(r)
        mm = r['min_margin_s']
        mm_str = f"{mm*1e3:+.4f} ms" if mm is not None else 'n/a'
        print(f"shot {k:04d}: row={r['row']} overflow={r['overflow']} "
              f"elapsed={r['elapsed_s']:.3f}s min_refill_margin={mm_str}")
    except Exception:
        print(f"shot {k} FAILED -- dumping hardware state:")
        dump_state(f"shot {k} exception")
        seq.mmio.write(SEQ_ENABLE, 0)
        raise

print()
print(f"Completed {len(results)}/{LOOP_COUNT} shots.")
all_margins = [r['min_margin_s'] for r in results if r['min_margin_s'] is not None]
if all_margins:
    print(f"Worst (smallest) refill margin seen across all shots: {min(all_margins)*1e3:+.4f} ms")
    if min(all_margins) < 0:
        print("At least one refill missed its modeled deadline. This is very likely the same "
              "failure mode as the TX-B hang -- widen that row's gap_s, or split the refill "
              "into smaller/more frequent chunks, before adding Ethernet back.")
    else:
        print("PASS: every refill in this run had positive margin against the modeled deadline. "
              "Safe to consider re-adding Ethernet (ideally on the separate RJ45 port) next.")

seq.mmio.write(SEQ_TTL_MODE, 0)
seq.mmio.write(SEQ_ENABLE, 0)
print('Done.')
